In [ ]:
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import scipy
import numpy as np
import xskillscore as xs

import dask
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor

import logging

import albedo_functions as af

In [ ]:
# Configurazione logging
log_file = "process_log.txt"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file),  # Write to file
    ]
)

In [ ]:
def dataset_season(dset,season):
    """
    SEASON SELECTION
    """
    if season == 'DJF':
        mon = [12,1,2]
    elif season == 'MAM':
        mon = [3,4,5]
    elif season == 'JJA':
        mon = [6,7,8]
    elif season == 'SON':
        mon = [9,10,11]
    elif season == 'Clim':
        mon = [11,12,1,2,3,4,5,6,7,8,9,10]
        
    dset = dset.sel(time=dset.time.dt.month.isin(mon))
    dset = dset.rolling(time = len(mon)).mean('time')
    dset = dset.sel(time = dset.time.dt.month == mon[-1])
    
    return dset

In [ ]:
def p_cal(ctrl, sens):
    a = ctrl.resample(time='YE').mean().var(dim='time')
    b = sens.resample(time='YE').mean().var(dim='time')

# Add a small constant to avoid division by zero
    epsilon = 1e-8
    a += epsilon
    b += epsilon

    F = a / b
    #df = len(ctrl.resample(time='YE').mean())
    df = ctrl.resample(time='YE').mean().sizes['time']
    p_value = scipy.stats.f.cdf(F, df, df)
    p_value = xr.Dataset({
        'p': xr.DataArray(p_value, dims=('lat', 'lon'),
                          coords={'lon': a['lon'],'lat': a['lat']})
    })
    return p_value

In [ ]:
def interannual_std(dset, n_bootstrap):
    """
    Calculate interannual standard deviation and its significance via bootstrapping.
    
    Parameters:
        dset (xarray.Dataset): Dataset containing 'time' dimension.
        varname (str): Variable name inside the dataset.
        n_bootstrap (int): Number of bootstrap iterations.
    
    Returns:
        xr.DataArray: observed_std
    """
    # Step 1: Group by year
    annual_mean = dset.groupby('time.year').mean('time')
    
    # Step 2: Observed std dev
    observed_std = annual_mean.std('year')
    
    # Step 3: Bootstrap resampling using xskillscore
    resampled = xs.resampling.resample_iterations_idx(
        annual_mean, n_bootstrap, dim='year', replace=True, dim_max=annual_mean.year.size
    )
    boot_std = resampled.std(dim='year')
    return observed_std

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
COV_PATH = f'{POST_DATA}/'
LAI_PATH = f'{CONFESS_DATA}/'
SAVE_PATH = f'{WORK_DIR}/'

In [ ]:
# Open datasets with Dask chunking
cvh_ctrl = xr.open_dataset(COV_PATH + f'{exp_ctrl}_effective_cvh_1x1.nc', chunks={'time': 12})['cvh']
cvl_ctrl = xr.open_dataset(COV_PATH + f'{exp_ctrl}_effective_cvl_1x1.nc', chunks={'time': 12})['cvl']
cvh_sens = xr.open_dataset(COV_PATH + f'{exp_sens}_effective_cvh_199311-201910_1x1.nc', chunks={'time': 12})['cvh']
cvl_sens = xr.open_dataset(COV_PATH + f'{exp_sens}_effective_cvl_199311-201910_1x1.nc', chunks={'time': 12})['cvl']

# Filter: remove low standard deviation data
cvh_ctrl = xr.where(cvh_ctrl.std('time') >= 0.0005, cvh_ctrl, np.nan)
cvh_sens = xr.where(cvh_sens.std('time') >= 0.0005, cvh_sens, np.nan)
cvl_ctrl = xr.where(cvl_ctrl.std('time') >= 0.0005, cvl_ctrl, np.nan)
cvl_sens = xr.where(cvl_sens.std('time') >= 0.0005, cvl_sens, np.nan)

a1ua_land_file = f'{BC_DATA}/'+exp_ctrl+'/icmcl_a1ua_regular_1x1.nc'
dset_a1ua = xr.open_dataset(a1ua_land_file, chunks=-1)

# select time from 1999
start_date = '1999-09-01'

cvh_ctrl = cvh_ctrl.sel(time=slice(start_date, None)).compute()
cvh_sens = cvh_sens.sel(time=slice(start_date, None)).compute()

cvl_ctrl = cvl_ctrl.sel(time=slice(start_date, None)).compute()
cvl_sens = cvl_sens.sel(time=slice(start_date, None)).compute()

In [ ]:
def processing_std(ctrl, sens, season, var):
    logging.info(f'Starting {season}')
    # Extract seasonal data
    seasonal_sens = dataset_season(sens,season)
    seasonal_ctrl = dataset_season(ctrl,season)

    # Compute interannual stds
    interannual_ctrl_std = af.land_seas_mask(interannual_std(seasonal_ctrl, 1000))
    interannual_sens_std = af.land_seas_mask(interannual_std(seasonal_sens, 1000))
    # Compute p-values
    p_value = af.land_seas_mask(p_cal(seasonal_ctrl,seasonal_sens))

    logging.info(f'Saving {season}')
    # Convert to Dataset and save
    interannual_ctrl_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a1ua_{var}_{season}_std_1999.nc')
    interannual_sens_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a52o_{var}_{season}_std_1999.nc')
    p_value.to_netcdf(f'{SAVE_PATH}delta_{var}_{season}_std_p_1999.nc')

In [ ]:
%%time
seasons=['DJF', 'JJA', 'MAM', 'SON']
futures=[]
logging.info("Starting parallel processing...")
with concurrent.futures.ProcessPoolExecutor() as executor:
    futures += [executor.submit(processing_std, cvh_ctrl, cvh_sens, i, 'cvh') for i in seasons]
    futures += [executor.submit(processing_std, cvl_ctrl, cvl_sens, i, 'cvl') for i in seasons]

    # Wait for all tasks to complete
    # progresso: stampa ogni task man mano che termina
    for _i, _f in enumerate(concurrent.futures.as_completed(futures), 1):
        _f.result()
        print(f"  completato {_i}/{len(futures)}", flush=True)